In [15]:
import pandas as pd
from pulp import *
import numpy as np

# Distance Matrix
dist_data = {
    'Location': ['Abdoun Branch', 'Shmesani Branch', 'AL-Rawda Branch', "Tla' Al-Ali Branch", 'Jabal AL-Hussein Branch', 'Shafa Badran Branch', 'Shmeisani Aramex Office', 'Jubeiha Aramex Office'],
    'Abdoun Branch': [0, 2.6, 5.5, 4.5, 3.5, 11.2, 3, 9.5],
    'Shmesani Branch': [2.6, 0, 4.1, 3.8, 1.6, 8.8, 0.2, 8.4],
    'AL-Rawda Branch': [5.5, 4.1, 0, 1.8, 5.3, 6.4, 4.9, 3.5],
    "Tla' Al-Ali Branch": [4.5, 3.8, 1.8, 0, 5.2, 8.1, 4, 4.6],
    'Jabal AL-Hussein Branch': [3.5, 1.6, 5.3, 5.2, 0, 9.1, 1.6, 10.1],
    'Shafa Badran Branch': [11.2, 8.8, 6.4, 8.1, 9.1, 0, 8.7, 6.1],
    'Shmeisani Aramex Office': [3, 0.2, 4.9, 4, 1.6, 8.7, 0, 8.3],
    'Jubeiha Aramex Office': [9.5, 8.4, 3.5, 4.6, 10.1, 6.1, 8.3, 0]
}
df_dist = pd.DataFrame(dist_data).set_index('Location')

# Order Data
orders_data = {
    'request_id': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    'pickup_branch': ['Abdoun Branch', 'AL-Rawda Branch', 'Shafa Badran Branch', 'Abdoun Branch', 'Shafa Badran Branch', 'Jabal AL-Hussein Branch', 'AL-Rawda Branch', 'Jabal AL-Hussein Branch', 'Jabal AL-Hussein Branch', 'AL-Rawda Branch'],
    'delivery_branch': ['Shmeisani Branch', 'Shmeisani Branch', 'Jabal AL-Hussein Branch', "Tla' Al-Ali Branch", 'Jabal AL-Hussein Branch', "Tla' Al-Ali Branch", 'Abdoun Branch', 'Shafa Badran Branch', 'Shmeisani Branch', 'Abdoun Branch'],
    'number_of_orders': [9, 6, 2, 10, 7, 9, 7, 9, 6, 6]
}
df_orders = pd.DataFrame(orders_data)


CAPACITY_PER_TRIP = 2
COST_PER_KM = 0.5

# Cleaning Data Mismatch
df_dist.index = df_dist.index.str.replace("Shmesani", "Shmeisani")
df_dist.columns = df_dist.columns.str.replace("Shmesani", "Shmeisani")

def get_distance(origin, dest):
    try:
        return df_dist.loc[origin, dest]
    except KeyError:
        return 9999



model = LpProblem("Aramex_Drone_Optimization", LpMinimize)
trip_vars = LpVariable.dicts("Trips", df_orders.index, lowBound=0, cat='Integer')

costs = []
for i in df_orders.index:
    pickup = df_orders.loc[i, 'pickup_branch']
    delivery = df_orders.loc[i, 'delivery_branch']
    dist = get_distance(pickup, delivery)
    costs.append(trip_vars[i] * dist * COST_PER_KM)

model += lpSum(costs)

for i in df_orders.index:
    demand = df_orders.loc[i, 'number_of_orders']
    model += trip_vars[i] * CAPACITY_PER_TRIP >= demand

status = model.solve()


print("="*60)
print("OPTIMIZATION RESULTS & ROUTING INTERPRETATION")
print("="*60)


# List of Depots
depots = ['Shmeisani Aramex Office', 'Jubeiha Aramex Office']

for i in df_orders.index:
    pickup = df_orders.loc[i, 'pickup_branch']
    delivery = df_orders.loc[i, 'delivery_branch']
    trips = int(trip_vars[i].value())


    d0_dist = get_distance(depots[0], pickup)
    d1_dist = get_distance(depots[1], pickup)

    if d0_dist <= d1_dist:
        start_depot = depots[0]
    else:
        start_depot = depots[1]

    # Print formatted block
    print(f"\n[Request ID: {df_orders.loc[i, 'request_id']}]")
    print(f"Route: {start_depot} -> {pickup} -> {delivery} -> {start_depot}")
    print(f"Trips Required: {trips}")

print("\n" + "*"*60)
print(f"Total Operational Cost: {value(model.objective)} JOD")
print("*"*60)

OPTIMIZATION RESULTS & ROUTING INTERPRETATION

[Request ID: 1]
Route: Shmeisani Aramex Office -> Abdoun Branch -> Shmeisani Branch -> Shmeisani Aramex Office
Trips Required: 5

[Request ID: 2]
Route: Jubeiha Aramex Office -> AL-Rawda Branch -> Shmeisani Branch -> Jubeiha Aramex Office
Trips Required: 3

[Request ID: 3]
Route: Jubeiha Aramex Office -> Shafa Badran Branch -> Jabal AL-Hussein Branch -> Jubeiha Aramex Office
Trips Required: 1

[Request ID: 4]
Route: Shmeisani Aramex Office -> Abdoun Branch -> Tla' Al-Ali Branch -> Shmeisani Aramex Office
Trips Required: 5

[Request ID: 5]
Route: Jubeiha Aramex Office -> Shafa Badran Branch -> Jabal AL-Hussein Branch -> Jubeiha Aramex Office
Trips Required: 4

[Request ID: 6]
Route: Shmeisani Aramex Office -> Jabal AL-Hussein Branch -> Tla' Al-Ali Branch -> Shmeisani Aramex Office
Trips Required: 5

[Request ID: 7]
Route: Jubeiha Aramex Office -> AL-Rawda Branch -> Abdoun Branch -> Jubeiha Aramex Office
Trips Required: 4

[Request ID: 8]
Ro